# SentinelAI — Notebook 04: MedDRA v29.0 EDA

**Goal:** Understand the structure and content of the MedDRA release files before loading them into PostgreSQL.

**Why this matters:**  
Module 2 of SentinelAI (the MedDRA Coding Engine) maps free-text MAUDE adverse event narratives to standardized MedDRA codes. Before building that pipeline, we need to know:
- What the MedDRA hierarchy looks like
- How many terms exist at each level
- What the data quality looks like (missing values, encoding issues, duplicates)
- Which files and fields we actually need

**Data source:** MedDRA v29.0 (MSSO Non-Commercial License, ID 33549)  
**Files location:** `../raw_data/MedDRA_29_0_English/MedAscii/`

---
## 1. MedDRA Terminology — What Do the Acronyms Mean?

MedDRA (Medical Dictionary for Regulatory Activities) is the international standard for classifying adverse events in medical device and drug regulation. It is maintained by the MSSO (MedDRA Maintenance and Support Services Organization) on behalf of the ICH (International Council for Harmonisation).

MedDRA organizes medical concepts in a **5-level hierarchy**, from most specific to most general:

| Acronym | Full Name | Level | Example |
|---------|-----------|-------|---------|
| **LLT** | Lowest Level Term | 1 (most specific) | *Insulin pump malfunction* |
| **PT**  | Preferred Term | 2 | *Device malfunction* |
| **HLT** | High Level Term | 3 | *Device issues* |
| **HLGT**| High Level Group Term | 4 | *General system disorders* |
| **SOC** | System Organ Class | 5 (most general) | *General disorders and administration site conditions* |

Additionally:
- **SMQ** (Standardised MedDRA Query): pre-built search baskets grouping related PTs for signal detection — e.g. all PTs related to "hypoglycaemia"
- **Primary SOC**: each PT belongs to exactly one *primary* SOC (the most clinically relevant one), but may appear under multiple SOCs as secondary assignments

### What SentinelAI needs and why

| Level | Used? | Why |
|-------|-------|-----|
| LLT | Input only | MAUDE narratives often describe symptoms at LLT granularity — we use LLTs as lookup targets during coding |
| **PT** | **Core output** | The PT is the standard reporting unit in pharmacovigilance. Module 2 outputs a PT code + confidence score. PRR/ROR signal detection in Module 3 runs on PTs |
| HLT / HLGT | Grouping | Used in the Grafana dashboard to group signals by organ system |
| **SOC** | Grouping | Top-level filter in the Signal Detection Dashboard |
| SMQ | Future | Not in v1 — useful for complex signal queries in a later iteration |

> **Rule of thumb:** We code to PT, group by SOC, and look up via LLT.

---
## 2. MedAscii File Overview

The MedDRA release package contains `.asc` files — plain text, `$`-delimited, no header row.

| File | Contents | Used by SentinelAI |
|------|----------|--------------------|
| `pt.asc` | All Preferred Terms | Reference |
| `llt.asc` | All Lowest Level Terms + PT mapping | Module 2 lookup |
| `hlt.asc` | High Level Terms | Reference |
| `hlt_pt.asc` | HLT → PT mapping table | Reference |
| `hlgt.asc` | High Level Group Terms | Reference |
| `hlgt_hlt.asc` | HLGT → HLT mapping | Reference |
| `soc.asc` | System Organ Classes | Reference |
| `soc_hlgt.asc` | SOC → HLGT mapping | Reference |
| **`mdhier.asc`** | **Full hierarchy in one file (PT + all levels)** | **Primary import file** |
| `smq_list.asc` | SMQ definitions | Future use |
| `smq_content.asc` | SMQ → PT memberships | Future use |
| `meddra_release.asc` | Version info | Metadata |

**Key insight:** `mdhier.asc` is the most important file — it contains one row per PT-SOC assignment with all hierarchy levels pre-joined. We filter to `primary_soc_flag = 'Y'` to get exactly one row per PT.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

MEDDRA_DIR = Path('../raw_data/MedDRA_29_0_English/MedAscii')
assert MEDDRA_DIR.exists(), f'MedDRA directory not found: {MEDDRA_DIR}'

# List all .asc files with sizes
files = sorted(MEDDRA_DIR.glob('*.asc'))
print(f'Found {len(files)} .asc files:\n')
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')

---
## 3. Load the Key Files

In [ ]:
def read_asc(path, columns):
    """Read a $-delimited MedDRA .asc file into a DataFrame."""
    rows = []
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\n').rstrip('$')
            if line:
                parts = line.split('$')
                # Pad or trim to expected column count
                parts = (parts + [''] * len(columns))[:len(columns)]
                rows.append(parts)
    return pd.DataFrame(rows, columns=columns)

In [ ]:
# --- mdhier.asc: full hierarchy (primary file) ---
# Columns per MedDRA format spec (dist_file_format_29_0_English.pdf)
mdhier = read_asc(MEDDRA_DIR / 'mdhier.asc', columns=[
    'pt_code', 'hlt_code', 'hlgt_code', 'soc_code',
    'pt_name', 'hlt_name', 'hlgt_name', 'soc_name',
    'soc_abbrev', 'null1', 'primary_soc_code', 'primary_soc_fg', 'null2'
])

# Cast codes to int
for col in ['pt_code', 'hlt_code', 'hlgt_code', 'soc_code']:
    mdhier[col] = pd.to_numeric(mdhier[col], errors='coerce')

print(f'mdhier.asc: {len(mdhier):,} rows (all PT-SOC assignments including secondary)')
mdhier.head(3)

In [ ]:
# --- Filter to primary SOC only (one row per PT) ---
mdhier_primary = mdhier[mdhier['primary_soc_fg'] == 'Y'].copy()
print(f'Primary SOC rows: {len(mdhier_primary):,}')
print(f'Non-primary (secondary SOC assignments): {len(mdhier) - len(mdhier_primary):,}')
print(f'\nUnique PTs in primary set: {mdhier_primary["pt_code"].nunique():,}')

In [ ]:
# --- llt.asc: Lowest Level Terms ---
llt = read_asc(MEDDRA_DIR / 'llt.asc', columns=[
    'llt_code', 'llt_name', 'pt_code',
    'llt_whoart_code', 'llt_harts_code', 'llt_costart_sym',
    'llt_icd9_code', 'llt_icd9cm_code', 'llt_icd10_code',
    'llt_currency', 'llt_jart_code', 'null1'
])

for col in ['llt_code', 'pt_code']:
    llt[col] = pd.to_numeric(llt[col], errors='coerce')

print(f'llt.asc: {len(llt):,} rows total')
print(f'  Current LLTs (llt_currency=Y): {(llt["llt_currency"]=="Y").sum():,}')
print(f'  Retired LLTs (llt_currency=N): {(llt["llt_currency"]=="N").sum():,}')
llt.head(3)

In [ ]:
# --- soc.asc: System Organ Classes ---
soc = read_asc(MEDDRA_DIR / 'soc.asc', columns=[
    'soc_code', 'soc_name', 'soc_abbrev',
    'soc_whoart_code', 'soc_harts_code', 'soc_costart_sym',
    'soc_icd9_code', 'soc_icd9cm_code', 'soc_icd10_code',
    'soc_jart_code'
])
soc['soc_code'] = pd.to_numeric(soc['soc_code'], errors='coerce')
print(f'System Organ Classes: {len(soc)}')
print(soc[['soc_code', 'soc_name', 'soc_abbrev']].to_string(index=False))

---
## 4. Hierarchy Size — How Many Terms at Each Level?

In [ ]:
summary = {
    'SOC  (System Organ Class)':       mdhier_primary['soc_code'].nunique(),
    'HLGT (High Level Group Term)':    mdhier_primary['hlgt_code'].nunique(),
    'HLT  (High Level Term)':          mdhier_primary['hlt_code'].nunique(),
    'PT   (Preferred Term)':           mdhier_primary['pt_code'].nunique(),
    'LLT  (Lowest Level Term, current)': (llt['llt_currency'] == 'Y').sum(),
    'LLT  (Lowest Level Term, total)': len(llt),
}

print('MedDRA v29.0 — Term counts per hierarchy level:')
print()
for level, count in summary.items():
    print(f'  {level:<42} {count:>7,}')

---
## 5. Distribution: PTs per SOC

This shows how unevenly terms are distributed across organ systems — important context for understanding class imbalance in the coding engine.

In [ ]:
pts_per_soc = (
    mdhier_primary
    .groupby('soc_name')['pt_code']
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'pt_code': 'pt_count'})
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=pts_per_soc, y='soc_name', x='pt_count', ax=ax, color='steelblue')
ax.set_title('MedDRA v29.0 — Number of PTs per SOC (primary SOC only)', fontsize=13)
ax.set_xlabel('Number of Preferred Terms')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(pts_per_soc.to_string(index=False))

---
## 6. Distribution: LLTs per PT

Most PTs have only a few LLTs. A PT that is also its own LLT (llt_code == pt_code) is called a **self-LLT** — this is common and expected in MedDRA.

In [ ]:
llt_current = llt[llt['llt_currency'] == 'Y'].copy()

llts_per_pt = llt_current.groupby('pt_code')['llt_code'].count()

print('LLTs per PT (current only):')
print(llts_per_pt.describe().round(1))

# Self-LLTs: llt_code == pt_code
self_llts = (llt_current['llt_code'] == llt_current['pt_code']).sum()
print(f'\nSelf-LLTs (llt_code == pt_code): {self_llts:,} ({self_llts/len(llt_current)*100:.1f}% of current LLTs)')
print('-> These are PTs that have no more specific LLT synonym.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
llts_per_pt.clip(upper=20).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='steelblue'
)
ax.set_title('Distribution of LLT count per PT (capped at 20)', fontsize=12)
ax.set_xlabel('Number of LLTs mapped to a PT')
ax.set_ylabel('Number of PTs')
plt.tight_layout()
plt.show()

---
## 7. Primary vs. Secondary SOC Assignments

A PT can appear under multiple SOCs (secondary assignments). For example, *Hyperglycaemia* may appear under *Metabolism* (primary) and *Endocrine* (secondary).  
We always use **primary SOC** for signal detection to avoid double-counting.

In [ ]:
soc_counts_per_pt = mdhier.groupby('pt_code')['soc_code'].nunique()

print('SOC assignments per PT:')
print(soc_counts_per_pt.value_counts().sort_index().rename('PT count'))

pct_multi = (soc_counts_per_pt > 1).mean() * 100
print(f'\nPTs with more than one SOC assignment: {pct_multi:.1f}%')
print('-> These all have exactly one primary SOC (flag=Y) — no ambiguity for signal detection.')

---
## 8. Text Quality Checks

In [ ]:
# --- Missing / empty names ---
print('=== Missing values in mdhier_primary ===')
print(mdhier_primary[['pt_name', 'hlt_name', 'hlgt_name', 'soc_name']]
      .isnull().sum())

empty_pt = (mdhier_primary['pt_name'].str.strip() == '').sum()
print(f'\nEmpty pt_name strings: {empty_pt}')

In [ ]:
# --- PT name length distribution ---
mdhier_primary['pt_name_len'] = mdhier_primary['pt_name'].str.len()

print('PT name length (characters):')
print(mdhier_primary['pt_name_len'].describe().round(1))

# Longest names
print('\nTop 5 longest PT names:')
print(mdhier_primary.nlargest(5, 'pt_name_len')[['pt_code', 'pt_name', 'pt_name_len']].to_string(index=False))

In [ ]:
# --- Non-ASCII characters in PT names ---
# Relevant because we use ASCII-only in Python scripts (see project convention)
non_ascii = mdhier_primary[mdhier_primary['pt_name'].apply(
    lambda x: any(ord(c) > 127 for c in str(x))
)]
print(f'PT names with non-ASCII characters: {len(non_ascii)}')
if len(non_ascii) > 0:
    print(non_ascii[['pt_code', 'pt_name']].head(10).to_string(index=False))

In [ ]:
# --- Duplicate PT names (different codes, same name) ---
dup_names = mdhier_primary[mdhier_primary.duplicated(subset='pt_name', keep=False)]
print(f'Duplicate PT names: {len(dup_names)}')
if len(dup_names) > 0:
    print(dup_names[['pt_code', 'pt_name']].sort_values('pt_name').head(10))

---
## 9. SMQ Overview (for future use)

SMQs (Standardised MedDRA Queries) are pre-built baskets of PTs grouped by clinical concept — e.g. all PTs related to *hypoglycaemia*, *embolic events*, or *device malfunction*.  
They are designed for **signal detection** across multiple related PTs simultaneously.  
SentinelAI will use SMQs in a later iteration to run broader signal queries.

In [ ]:
# --- smq_list.asc: available SMQs ---
smq_list = read_asc(MEDDRA_DIR / 'smq_list.asc', columns=[
    'smq_code', 'smq_name', 'smq_level', 'smq_description',
    'smq_source', 'smq_note', 'MedDRA_version', 'status',
    'smq_algorithm'
])

print(f'Total SMQs available: {len(smq_list)}')
print(f'Active SMQs: {(smq_list["status"] == "A").sum()}')
print()

# Device-related SMQs
device_smqs = smq_list[smq_list['smq_name'].str.contains('device|malfunction|implant', case=False, na=False)]
print(f'Potentially relevant SMQs (device/malfunction/implant):')
print(device_smqs[['smq_code', 'smq_name', 'status']].to_string(index=False))

---
## 10. What We Load Into PostgreSQL — Summary

Based on this EDA, here is what `import_meddra.py` does:

| Table | Source | Filter | Rows (approx.) |
|-------|--------|--------|----------------|
| `processed.meddra_terms` | `mdhier.asc` | `primary_soc_fg = 'Y'` | ~27,000 |
| `processed.meddra_llt` | `llt.asc` | `llt_currency = 'Y'` | ~70,000 |

**Not loaded in v1 (but available):**
- SMQ tables — planned for Phase 3 signal enhancement
- `meddra_history_english.asc` — term change history, useful for version migration later
- Secondary SOC assignments — not needed since signal detection uses primary SOC only

**Next step:** Run `import_meddra.py` against the Hetzner PostgreSQL instance, then build the PubMedBERT embedding pipeline for all PT names.

In [ ]:
# Final sanity check: preview what will land in processed.meddra_terms
import_preview = mdhier_primary[[
    'pt_code', 'pt_name',
    'hlt_code', 'hlt_name',
    'hlgt_code', 'hlgt_name',
    'soc_code', 'soc_name'
]].copy()

print(f'Rows to import into processed.meddra_terms: {len(import_preview):,}')
print()
import_preview.head(5)